# Data Cleaning

## =>Objective
The purpose of this notebook is to clean the Olist datasets by handling duplicates, missing values, incorrect data types, inconsistent text formatting, and other data quality issues. The cleaned datasets will serve as the foundation for feature engineering, SQL analysis, statistical analysis, and Power BI dashboards.

In [2]:
import pandas as pd
import numpy as np
import os

print(os.getcwd())

# Display all columns
pd.set_option('display.max_columns', None)

# Display wider text
pd.set_option('display.max_colwidth', None)

C:\Users\91789\Desktop\Customer Growth & Experimentation Analytics\03_Python_ETL\Notebooks


In [3]:
customers = pd.read_csv("../../01_Raw_Data/olist_customers_dataset.csv")
orders = pd.read_csv("../../01_Raw_Data/olist_orders_dataset.csv")
order_items = pd.read_csv("../../01_Raw_Data/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../../01_Raw_Data/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../../01_Raw_Data/olist_order_reviews_dataset.csv")
products = pd.read_csv("../../01_Raw_Data/olist_products_dataset.csv")
sellers = pd.read_csv("../../01_Raw_Data/olist_sellers_dataset.csv")
geolocation = pd.read_csv("../../01_Raw_Data/olist_geolocation_dataset.csv")
product_category = pd.read_csv("../../01_Raw_Data/product_category_name_translation.csv")

In [4]:
customers_clean = customers.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
order_payments_clean = order_payments.copy()
order_reviews_clean = order_reviews.copy()
products_clean = products.copy()
sellers_clean = sellers.copy()
geolocation_clean = geolocation.copy()
product_category_clean = product_category.copy()

## Customers Table Cleaning

In [4]:
#Checking the Shape
customers_clean.shape

(99441, 5)

In [5]:
#Checking for Duplicate Rows
customers_clean.duplicated().sum()

0

In [6]:
#Checking Missing Values
customers_clean.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [7]:
#Verifying Data Types
customers_clean.dtypes

customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

In [8]:
#Checking for Invalid Values for all columns: 
customers_clean["customer_id"].str.len().describe()

count    99441.0
mean        32.0
std          0.0
min         32.0
25%         32.0
50%         32.0
75%         32.0
max         32.0
Name: customer_id, dtype: float64

In [9]:
customers_clean["customer_unique_id"].str.len().describe()

count    99441.0
mean        32.0
std          0.0
min         32.0
25%         32.0
50%         32.0
75%         32.0
max         32.0
Name: customer_unique_id, dtype: float64

In [10]:
customers_clean["customer_zip_code_prefix"].describe()

count    99441.000000
mean     35137.474583
std      29797.938996
min       1003.000000
25%      11347.000000
50%      24416.000000
75%      58900.000000
max      99990.000000
Name: customer_zip_code_prefix, dtype: float64

In [11]:
customers_clean["customer_state"].value_counts().sort_index()

customer_state
AC       81
AL      413
AM      148
AP       68
BA     3380
CE     1336
DF     2140
ES     2033
GO     2020
MA      747
MG    11635
MS      715
MT      907
PA      975
PB      536
PE     1652
PI      495
PR     5045
RJ    12852
RN      485
RO      253
RR       46
RS     5466
SC     3637
SE      350
SP    41746
TO      280
Name: count, dtype: int64

In [12]:
customers_clean["customer_city"].head(20)

0                    franca
1     sao bernardo do campo
2                 sao paulo
3           mogi das cruzes
4                  campinas
5            jaragua do sul
6                 sao paulo
7                   timoteo
8                  curitiba
9            belo horizonte
10            montes claros
11           rio de janeiro
12         lencois paulista
13                sao paulo
14            caxias do sul
15               piracicaba
16           rio de janeiro
17                guarulhos
18                sao paulo
19                   pacaja
Name: customer_city, dtype: object

In [13]:
#Standardizing Text
customers_clean["customer_city"] = (
    customers_clean["customer_city"]
    .str.strip()
)

## Customers Table (Conclusion)

- Duplicate rows: 0
- Missing values: None
- Data types: Verified
- Invalid values: None
- Text standardization: Not required
- Final status: No cleaning required

In [14]:
customers_clean.to_csv("../Output/customers_clean.csv", index=False)

## Orders Table Cleaning

In [15]:
#Checking Shape
orders_clean.shape

(99441, 8)

In [16]:
#Check for Duplicate Rows
orders_clean.duplicated().sum()

0

In [17]:
#Checking for Missing Values
orders_clean.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [18]:
#Verifying Data Types
orders_clean.dtypes

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

In [19]:
#Convert Timestamp Columns
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders_clean[date_columns] = orders_clean[date_columns].apply(pd.to_datetime)

In [20]:
#Check for Primary Key
orders_clean["order_id"].duplicated().sum()

0

In [21]:
#Verifying Order Status
orders_clean["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [22]:
orders_clean.groupby("order_status").agg({
    "order_approved_at": lambda x: x.isnull().sum(),
    "order_delivered_carrier_date": lambda x: x.isnull().sum(),
    "order_delivered_customer_date": lambda x: x.isnull().sum()
})

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


In [23]:
orders_clean.to_csv("../Output/orders_clean.csv", index=False)

## Orders Table (Conclusion)

### Cleaning Performed
- Verified duplicate rows (none found).
- Converted all timestamp columns to datetime.
- Verified unique order IDs.
- Validated all order status categories.

### Data Quality Validation
- Missing timestamps are consistent with the order lifecycle for non-completed orders.
- A small number of delivered orders have missing timestamps.
- These records were retained because the correct values cannot be inferred.

### Final Action
- No rows removed.
- No values imputed.
- Timestamp columns converted to datetime.

## Order_items Table Cleaning

In [24]:
#Checking Shape
order_items_clean.shape

(112650, 7)

In [25]:
#Check for Duplicate Rows
order_items_clean.duplicated().sum()

0

In [26]:
#Checking for Missing Values
order_items_clean.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [27]:
#Verifying Data Types
order_items_clean.dtypes

order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

In [28]:
# Converting datatype of Shipping Date
order_items_clean["shipping_limit_date"] = pd.to_datetime(
    order_items_clean["shipping_limit_date"]
)

In [29]:
# Validating Primary Key : As per the audit there was no Primary key, we are making a check for composite key 
order_items_clean.duplicated(subset=["order_id", "order_item_id"]).sum()

0

In [30]:
# Validating Price values, making sure no negative values are present
order_items_clean["price"].describe()

count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max        6735.000000
Name: price, dtype: float64

In [31]:
(order_items_clean["price"] <= 0).sum()

0

In [32]:
# Validating Freight Price values, making sure no negative values are present
order_items_clean["freight_value"].describe()

count    112650.000000
mean         19.990320
std          15.806405
min           0.000000
25%          13.080000
50%          16.260000
75%          21.150000
max         409.680000
Name: freight_value, dtype: float64

In [33]:
(order_items_clean["freight_value"] < 0).sum()

0

In [34]:
order_items_clean["order_item_id"].value_counts().sort_index()

order_item_id
1     98666
2      9803
3      2287
4       965
5       460
6       256
7        58
8        36
9        28
10       25
11       17
12       13
13        8
14        7
15        5
16        3
17        3
18        3
19        3
20        3
21        1
Name: count, dtype: int64

## Order Items Table

### Cleaning Performed
- Verified duplicate rows (none found).
- Validated composite key (order_id + order_item_id).
- Converted shipping_limit_date to datetime.
- Verified price and freight values.

### Data Quality Validation
- No missing values.
- No invalid prices.
- No negative freight charges.

### Final Action
- Converted shipping_limit_date to datetime.
- No rows removed.
- No values imputed.

In [35]:
order_items_clean.to_csv("../Output/order_items_clean.csv", index=False)

## Order_payments Table Cleaning

In [36]:
#Checking Shape
order_payments_clean.shape

(103886, 5)

In [37]:
#Check for Duplicate Rows
order_payments_clean.duplicated().sum()

0

In [38]:
#Checking for Missing Values
order_payments_clean.isnull().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [39]:
#Verifying Data Types
order_payments_clean.dtypes

order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

In [40]:
# Validating Composite key: As per the audit no primary key is present in the table
order_payments_clean.duplicated(subset = ['order_id','payment_sequential']).sum()

0

In [41]:
# Verifying payment types
order_payments_clean["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [42]:
order_payments_clean[
    order_payments_clean["payment_type"] == "not_defined"
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [43]:
orders_clean[orders_clean['order_id'] == '4637ca194b6387e2d538dc89b124b0ee']

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
39919,4637ca194b6387e2d538dc89b124b0ee,a73c1f73f5772cf801434bf984b0b1a7,canceled,2018-09-03 14:14:25,NaT,NaT,NaT,2018-09-10


In [44]:
# Qaulity check 
order_payments_clean["payment_installments"].describe()


count    103886.000000
mean          2.853349
std           2.687051
min           0.000000
25%           1.000000
50%           1.000000
75%           4.000000
max          24.000000
Name: payment_installments, dtype: float64

In [45]:
(order_payments_clean["payment_installments"] <= 0).sum()

2

In [46]:
order_payments_clean[order_payments_clean['payment_installments'] == 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [47]:
order_payments_clean["payment_value"].describe()

count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64

In [48]:
order_payments_clean[order_payments_clean['payment_value'] == 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [49]:
order_payments_clean[
    order_payments_clean["order_id"].isin([
        "744bade1fcf9ff3f31d860ace076d422",
        "1a57108394169c0b47d8f876acc9ba2d"
    ])
].sort_values(["order_id", "payment_sequential"])

,order_id,payment_sequential,payment_type,payment_installments,payment_value
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


In [50]:
(order_payments_clean["payment_value"] <= 0).sum()

9

In [51]:
order_payments_clean["payment_sequential"].value_counts().sort_index()

payment_sequential
1     99360
2      3039
3       581
4       278
5       170
6       118
7        82
8        54
9        43
10       34
11       29
12       21
13       13
14       10
15        8
16        6
17        6
18        6
19        6
20        4
21        4
22        3
23        2
24        2
25        2
26        2
27        1
28        1
29        1
Name: count, dtype: int64

In [52]:
zero_payment_orders = order_payments_clean.loc[
    order_payments_clean["payment_value"] == 0,
    "order_id"
].unique()

order_payments_clean[
    order_payments_clean["order_id"].isin(zero_payment_orders)
].sort_values(["order_id", "payment_sequential"])

,order_id,payment_sequential,payment_type,payment_installments,payment_value
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
33781,45ed6e85398a87c253db47c2d9f48216,1,voucher,1,21.13
11755,45ed6e85398a87c253db47c2d9f48216,2,voucher,1,50.01
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
40546,6ccb433e00daae1283ccc956189c82ae,1,credit_card,5,84.67
93478,6ccb433e00daae1283ccc956189c82ae,2,voucher,1,14.65
92318,6ccb433e00daae1283ccc956189c82ae,3,voucher,1,22.72
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
20963,8bcbe01d44d147f901cd3192671144db,1,credit_card,1,36.21


In [53]:
order_payments_clean.loc[
    order_payments_clean["payment_value"] == 0,
    ["order_id", "payment_value"]
]

,order_id,payment_value
19922,8bcbe01d44d147f901cd3192671144db,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,0.0
43744,6ccb433e00daae1283ccc956189c82ae,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,0.0
57411,00b1cb0320190ca0daa2c88b35206009,0.0
62674,45ed6e85398a87c253db47c2d9f48216,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,0.0
94427,c8c528189310eaa44a745b8d9d26908b,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,0.0


## Order Payments Table

### Cleaning Performed
- Verified duplicate rows (none found).
- Verified composite key (order_id + payment_sequential).
- Validated payment methods.
- Validated payment values.
- Validated installment values.

### Data Quality Validation
- Three records contain `payment_type = not_defined`; these correspond to canceled orders and were retained.
- Nine payment records have a payment value of zero; investigation showed these are valid payment sequence records or canceled orders.
- Two credit card payments have `payment_installments = 0`. Since the correct installment count cannot be determined, these records were retained and documented.

### Final Action
- No rows removed.
- No values imputed.
- No data types changed.

In [54]:
order_payments_clean.to_csv("../Output/order_payments_clean.csv", index=False)

## Order_reviews Table Cleaning

In [55]:
#Checking Shape
order_reviews_clean.shape

(99224, 7)

In [56]:
#Check for Duplicate Rows
order_reviews_clean.duplicated().sum()

0

In [57]:
#Checking for Missing Values
order_reviews_clean.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [58]:
#Verifying Data Types
order_reviews_clean.dtypes

review_id                  object
order_id                   object
review_score                int64
review_comment_title       object
review_comment_message     object
review_creation_date       object
review_answer_timestamp    object
dtype: object

In [59]:
# Converting to datetime datatype
date_cols = [
    "review_creation_date",
    "review_answer_timestamp"
]

order_reviews_clean[date_cols] = order_reviews_clean[date_cols].apply(pd.to_datetime)

In [60]:
#Validating Review Score
order_reviews_clean["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [61]:
#No Primary key as per the audit, however composite key is found(review_id + order_id)
order_reviews_clean.duplicated(
    subset=["review_id", "order_id"]
).sum()

0

## Order Reviews Table

### Cleaning Performed
- Verified duplicate rows (none found).
- Converted review_creation_date and review_answer_timestamp to datetime.
- Validated review scores.
- Verified composite key (review_id + order_id).

### Data Quality Validation
- Missing review titles and review comments were retained because these fields are optional and depend on customer input.
- No invalid review scores detected.

### Final Action
- Converted datetime columns.
- No rows removed.
- No values imputed.

In [62]:
order_reviews_clean.to_csv("../Output/order_reviews_clean.csv",index=False)

## Products Table Cleaning

In [63]:
#Checking Shape
products_clean.shape

(32951, 9)

In [64]:
#Check for Duplicate Rows
products_clean.duplicated().sum()

0

In [65]:
#Checking for Missing Values
products_clean.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [66]:
#Verifying Data Types
products_clean.dtypes

product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

In [67]:
#Checking for Primary key 
products_clean["product_id"].duplicated().sum()

0

In [68]:
#Category Validation
products_clean["product_category_name"].value_counts(dropna=False)

product_category_name
cama_mesa_banho                  3029
esporte_lazer                    2867
moveis_decoracao                 2657
beleza_saude                     2444
utilidades_domesticas            2335
                                 ... 
fashion_roupa_infanto_juvenil       5
casa_conforto_2                     5
pc_gamer                            3
seguros_e_servicos                  2
cds_dvds_musicais                   1
Name: count, Length: 74, dtype: int64

In [69]:
#Checking for incorrect values(-ve values)
products_clean.describe()

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000


In [70]:
products_clean[
    products_clean["product_category_name"].isnull()
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


In [71]:
products_clean[
    products_clean["product_weight_g"] == 0
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


## Products Table

### Cleaning Performed
- Verified there are no duplicate rows.
- Verified `product_id` is unique.
- Validated data types.

### Data Quality Validation
- 610 products have missing catalog metadata (`product_category_name`, `product_name_lenght`, `product_description_lenght`, `product_photos_qty`).
  Physical attributes are present for these products, indicating missing catalog information rather than missing product records.
- 2 products have missing physical measurements (`product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm`).
- 4 products have a recorded weight of 0 g, which is likely a data entry anomaly. Since the correct values cannot be determined, these records were retained.

### Final Action
- No rows removed.
- No missing values imputed.
- No business values modified.

In [73]:
products_clean.to_csv("../Output/products_clean.csv",index=False)

## Sellers Table cleaning

In [74]:
#Checking Shape
sellers_clean.shape

(3095, 4)

In [75]:
#Check for Duplicate Rows  
sellers_clean.duplicated().sum()

0

In [76]:
#Checking for Missing Values
sellers_clean.isnull().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [77]:
#Verifying Data Types
sellers_clean.dtypes

seller_id                 object
seller_zip_code_prefix     int64
seller_city               object
seller_state              object
dtype: object

In [78]:
#Checking for Primary key 
sellers_clean["seller_id"].duplicated().sum()

0

In [80]:
#Validate ZIP Code Length
sellers_clean["seller_zip_code_prefix"].describe()

count     3095.000000
mean     32291.059451
std      32713.453830
min       1001.000000
25%       7093.500000
50%      14940.000000
75%      64552.500000
max      99730.000000
Name: seller_zip_code_prefix, dtype: float64

In [81]:
#Checking State Codes
sellers_clean["seller_state"].value_counts().sort_index()

seller_state
AC       1
AM       1
BA      19
CE      13
DF      30
ES      23
GO      40
MA       1
MG     244
MS       5
MT       4
PA       1
PB       6
PE       9
PI       1
PR     349
RJ     171
RN       5
RO       2
RS     129
SC     190
SE       2
SP    1849
Name: count, dtype: int64

In [82]:
sellers_clean.to_csv("../output/sellers_clean.csv", index = False)

# Sellers Table

## Table Information

**Table Name:** sellers

**Table Grain:** One row represents one unique seller registered on the Olist platform.

**Primary Key:** seller_id

---

## Cleaning Performed

- Verified table shape.
- Verified there are no duplicate rows.
- Verified `seller_id` is unique.
- Checked for missing values across all columns.
- Validated data types for each column.
- Reviewed seller ZIP code prefixes for reasonable values.
- Validated seller state codes.

---

## Data Quality Validation

### Duplicate Rows
- No duplicate rows found.

### Primary Key
- `seller_id` is unique for every record.
- No duplicate primary keys detected.

### Missing Values
- No missing values found in any column.

### Data Types
- All columns have appropriate data types.
- No datatype conversions were required.

### Seller Location Information
- Seller ZIP code prefixes are populated for all records.
- Seller state codes contain valid Brazilian state abbreviations.
- No inconsistencies requiring correction were identified.

---

## Final Action

- No rows removed.
- No values modified.
- No missing values imputed.
- Dataset retained in its original form after validation.

## Geolocation Table Cleaning

In [83]:
#Checking Shape
geolocation_clean.shape

(1000163, 5)

In [91]:
#Check for Duplicate Rows  
geolocation_clean.duplicated().sum()

261831

In [85]:
#Checking for Missing Values
geolocation_clean.isnull().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [86]:
#Verifying Data Types
geolocation_clean.dtypes

geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

In [87]:
#ZIP Code Prefix
geolocation_clean["geolocation_zip_code_prefix"].duplicated().sum()

981148

In [88]:
#Number of Unique ZIP Codes
geolocation_clean["geolocation_zip_code_prefix"].nunique()

19015

In [94]:
#Duplicate Investigation
geolocation_clean[
    geolocation_clean.duplicated()
].head(20)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
15,1046,-23.546081,-46.644820,sao paulo,SP
44,1046,-23.546081,-46.644820,sao paulo,SP
65,1046,-23.546081,-46.644820,sao paulo,SP
66,1009,-23.546935,-46.636588,sao paulo,SP
67,1046,-23.546081,-46.644820,sao paulo,SP
72,1046,-23.545320,-46.644069,sao paulo,SP
79,1050,-23.549854,-46.643139,sao paulo,SP
80,1032,-23.540775,-46.635515,sao paulo,SP
82,1046,-23.546081,-46.644820,sao paulo,SP
86,1048,-23.547449,-46.640169,são paulo,SP


In [92]:
#City Name Variations
geolocation_clean["geolocation_city"].nunique()

8011

In [93]:
#State Codes
geolocation_clean["geolocation_state"].value_counts().sort_index()

geolocation_state
AC      1301
AL      4183
AM      2432
AP       853
BA     36045
CE     11674
DF     12986
ES     16748
GO     20139
MA      7853
MG    126336
MS     10431
MT     12031
PA     10853
PB      5538
PE     16432
PI      4549
PR     57859
RJ    121169
RN      5041
RO      3478
RR       646
RS     61851
SC     38328
SE      3563
SP    404268
TO      3576
Name: count, dtype: int64

In [95]:
#Dropping duplicates
geolocation_clean = geolocation_clean.drop_duplicates().reset_index(drop=True)

In [96]:
#Verification for the duplicates cleaned or not 
geolocation_clean.duplicated().sum()

0

In [104]:
geolocation_clean.to_csv("../output/geolocation_clean.csv", index = False)

# Geolocation Table

## Table Information

**Table Name:** geolocation

**Table Grain:** One row represents a recorded geolocation observation (latitude and longitude) associated with a ZIP code prefix.

---

## Cleaning Performed

- Verified table shape.
- Verified missing values.
- Verified data types.
- Investigated duplicate rows.
- Validated ZIP code prefixes.
- Reviewed city and state values.

---

## Data Quality Validation

### Duplicate Rows
- Found 261,831 exact duplicate rows.
- Duplicate records contained identical values across all columns.
- These duplicates did not provide additional information and were removed.

### Missing Values
- No missing values found in any column.

### Data Types
- All columns have appropriate data types.
- No datatype conversions were required.

### ZIP Code Prefix
- ZIP code prefixes are intentionally repeated because multiple geolocation observations may exist for the same prefix.
- These records were retained.

### City Names
- Minor spelling variations (e.g., accented vs. unaccented city names) were observed.
- No modifications were made during cleaning to preserve the original data.
- Standardization can be performed later during feature engineering if required.

---

## Final Action

- Removed exact duplicate rows.
- No missing values imputed.
- No business values modified.
- Preserved all valid geolocation observations.

## Product_category Table Cleaning

In [97]:
#Checking Shape
product_category_clean.shape

(71, 2)

In [5]:
product_category_clean.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [98]:
#Check for Duplicate Rows  
product_category_clean.duplicated().sum()

0

In [99]:
#Checking for Missing Values
product_category_clean.isnull().sum()

product_category_name            0
product_category_name_english    0
dtype: int64

In [100]:
#Verifying Data Types
product_category_clean.dtypes

product_category_name            object
product_category_name_english    object
dtype: object

In [102]:
#Portuguese Category Uniqueness
product_category_clean["product_category_name"].duplicated().sum()

0

In [103]:
#English Category Uniqueness
product_category_clean["product_category_name_english"].duplicated().sum()

0

In [6]:
product_category_clean.to_csv("../output/product_category_clean.csv", index = False)

# Product Category Translation Table

## Table Information

**Table Name:** product_category_name_translation

**Table Grain:** One row represents the mapping of a single Portuguese product category to its English translation.

---

## Cleaning Performed

- Verified table shape.
- Verified duplicate rows.
- Checked for missing values.
- Validated data types.
- Verified uniqueness of Portuguese category names.
- Verified uniqueness of English category names.

---

## Data Quality Validation

### Duplicate Rows
- No duplicate rows found.

### Missing Values
- No missing values found in any column.

### Data Types
- Both columns have the correct `object` data type.
- No datatype conversions were required.

### Category Mapping
- Each Portuguese category maps to exactly one English category.
- Each English category has a unique corresponding Portuguese category.
- No inconsistent or duplicate mappings were identified.

---

## Final Action

- No rows removed.
- No values modified.
- No missing values imputed.
- Dataset retained in its original form after validation.